# Differential expression analysis


You have run the nf-core/rnaseq pipeline and checked the first quality control metrics of your fastq files. This was, however, only the primary analysis and we want to take it further.

Due to the computational demand of the pipeline, you only ran the pipeline on two of the 16 samples in the study yesterday. We provide you an essential output of nf-core/rnaseq pipeline in the `data` folder: It contains the combined epression matrix as produced by Salmon, which provides transcript levels for each gene (rows) and each sample (columns).


We would now like to understand exactly the difference between the expression in our groups of mice. 
Which pipeline would you use for this?

`nf-core/differentialabundance` is built exactly for this step: taking an already-quantified expression matrix (like the Salmon output we were given) and testing for statistically significant differences in expression between defined sample groups. It wraps DESeq2 (for RNA-seq count data) internally, handles normalization, generates the contrasts you define (the genotype/condition comparisons), and produces the diagnostic plots you made earlier (PCA, boxplots) plus the differential expression results and report.

Have a close look at the pipeline's "Usage" page on the [nf-core docs](https://nf-co.re). You will need to create a samplesheet (based on the column names in the provided matrix).

In [8]:
import pandas as pd

counts = pd.read_csv("data/salmon.merged.gene_counts.tsv", sep="\t")
samples = counts.columns.drop(["gene_id", "gene_name"]).tolist()

new_df = pd.DataFrame({"sample": samples})
new_df["Condition"] = new_df["sample"].str.rsplit("_", n=1).str[0]  # drop trailing _1/_2/_3/_4

new_df.to_csv("samplesheet.csv", index=False)
print(new_df)

        sample Condition
0   Sham_oxy_1  Sham_oxy
1   Sham_oxy_2  Sham_oxy
2   Sham_oxy_3  Sham_oxy
3   Sham_oxy_4  Sham_oxy
4   Sham_Sal_1  Sham_Sal
5   Sham_Sal_2  Sham_Sal
6   Sham_Sal_3  Sham_Sal
7   Sham_Sal_4  Sham_Sal
8    SNI_oxy_1   SNI_oxy
9    SNI_oxy_2   SNI_oxy
10   SNI_oxy_3   SNI_oxy
11   SNI_oxy_4   SNI_oxy
12   SNI_Sal_1   SNI_Sal
13   SNI_Sal_2   SNI_Sal
14   SNI_Sal_3   SNI_Sal
15   SNI_Sal_4   SNI_Sal


Please paste here the command you used. You may need to inspect the provided expression matrix more closely and create additional files, like a samplesheet (based on the column names) or a contrast file (there happens to also be one in `data/` that you can use).

In [12]:
!nextflow run nf-core/differentialabundance -r 2.0.0 \
  -profile docker \
  --input samplesheet.csv \
  --contrasts data/contrasts.csv \
  --matrix data/salmon.merged.gene_counts.tsv \
  --feature-length-matrix data/rnaseq-out-small/salmon.merged.gene_counts_length_scaled.tsv \
  --genome GRCm38 \
  --outdir results


 N E X T F L O W   ~  version 26.04.6

Launching `https://github.com/nf-core/differentialabundance` [ecstatic_kay] revision: 30ed7741fc [2.0.0]

WARN: Unrecognized config option 'validation.defaultIgnoreParams'
WARN: Unrecognized config option 'validation.monochromeLogs'

------------------------------------------------------
                                        ,--./,-.
        ___     __   __   __   ___     /,-._.--~'
  |\ | |__  __ /  ` /  \ |__) |__         }  {
  | \| |       \__, \__/ |  \ |___     \`-._,-`-,
                                        `._,._,'
  nf-core/differentialabundance 2.0.0
------------------------------------------------------

[base] Input/output options
  input                       : samplesheet.csv
  contrasts                   : data/contrasts.csv
  outdir                      : results

[base] Abundance values
  matrix                      : data/salmon.merged.gene_counts.tsv

[preprocessing] Filtering
  filtering_min_abundance     : 1
  filtering_

Explain all the parameters you set and why you set them in this way. If you used or created additional files as input, explain what they are used for.

- `-r 2.0.0`: pins the pipeline to a specific, tagged release rather than whatever is on the default branch. This is what makes the analysis reproducible, anyone rerunning this command later gets the exact same pipeline code, module versions, and default parameters, rather than picking up unrelated changes made to the pipeline since.
- `-profile docker`: tells Nextflow to run every process inside its designated container image rather than expecting tools to be installed locally. This guarantees the exact software versions the pipeline authors intended, and avoids the "tool not found on PATH" problems from earlier in this course (like the missing gtf2featureAnnotation.R script), since the container ships everything needed.
- `--input samplesheet.csv`: points to the sample metadata table, generated from the column names of the expression matrix. It maps each sample name to its experimental group (in this case, a combined Condition column with values like SNI_Sal, SNI_oxy, Sham_Sal, Sham_oxy). The pipeline uses this to know which samples belong to which group when computing group-wise comparisons.
- `--contrasts data/contrasts.csv`: defines which comparisons to actually test statistically, in this case, SNI_Sal vs SNI_oxy, and Sham_Sal vs Sham_oxy, specifying the variable (Condition), and which group is the reference versus the target for each contrast. Without this, the pipeline wouldn't know which pairwise comparisons to run among the four groups.
- `--matrix data/salmon.merged.gene_counts.tsv`: the pre-computed gene-level count matrix from Salmon (via nf-core/rnaseq), one row per gene, one column per sample. This is the actual expression data the differential analysis is performed on, so the pipeline skips re-quantification and goes straight to normalization and statistical testing.
- `--gene_length_matrix`: supplies per-sample, per-gene effective length estimates from Salmon, alongside the raw counts matrix. This allows the pipeline's internal DESeq2 step to properly account for gene-length-driven differences in read counts during normalization, rather than assuming uniform gene length across the dataset.
- `--genome GRCm38`: specifies the mouse reference genome build used for quantification (matching what was used in the upstream nf-core/rnaseq run). This lets the pipeline automatically retrieve the matching GTF annotation via AWS-iGenomes, needed to translate gene IDs into gene symbols/descriptions for the report and plots (this replaced manually supplying --gtf with a downloaded file, since specifying the genome build directly is simpler and less error-prone than matching Ensembl release versions by hand).
- `--outdir results`: where all outputs (QC plots, PCA, differential expression tables, the final HTML/Quarto report) are written.

What were the outputs of the pipeline?

The pipeline produced several categories of output: **an interactive HTML report** (`study_differentialabundance_report.html`) summarizing the whole analysis, per-contrast **differential expression tables** (raw, significance-filtered, and gene-annotated versions from DESeq2), **processed expression matrices** (size-factor normalized counts and variance-stabilized values), a set of *QC and diagnostic plots** (PCA, boxplots, sample dendrogram, dispersion estimates, volcano plots per contrast), an **interactive Shiny app** for exploring the results in R, intermediate **DESeq2 objects and normalization factors** for reproducibility, and **pipeline execution metadata** (run report, timeline, DAG, software versions).

In [2]:
#!TODO

Would you exclude any samples? If yes, which and why?

![dendrogram](results/plots/exploratory/contrasts/Condition/png/sample_dendrogram.png) \
I would exclude **SNI_Sal_2** and **SNI_Sal_4**, based on both the sample dendrogram (they form an isolated branch, distant from all other samples including their own replicates) and the PCA (they're extreme outliers on PC1, while SNI_Sal_1 and SNI_Sal_3 cluster normally with the rest).

How many genes were differentially expressed in each contrast? Does this confirm what the paper mentions?

In [36]:
!wc -l results/tables/differential/contrasts/condition_control_treated_study.deseq2.results_filtered.tsv
!wc -l results/tables/differential/contrasts/condition_control_treated_test_study.deseq2.results_filtered.tsv

19 results/tables/differential/contrasts/condition_control_treated_study.deseq2.results_filtered.tsv
8 results/tables/differential/contrasts/condition_control_treated_test_study.deseq2.results_filtered.tsv


Subtract 1 for the header line.\
So `condition_control_treated_study.deseq2.results_filtered.tsv` has 18 differentially expressed and \
`condition_control_treated_test_study.deseq2.results_filtered.tsv` has 7 differentially expressed.

In [ ]:
!nextflow run nf-core/differentialabundance -r 2.0.0 \
  -profile docker \
  --input samplesheet.csv \
  --contrasts data/contrasts2.csv \
  --matrix data/salmon.merged.gene_counts.tsv \
  --feature-length-matrix data/rnaseq-out-small/salmon.merged.gene_counts_length_scaled.tsv \
  -params-file params-file.json \
  --genome GRCm38 \
  --outdir results_match_paper \
  -resume

I recreated the report with the same conditions as in the paper, including `max_pval` and `min_fold_change`.

|                                     |  up | down | sum |
|:-----------------------------------:|:---:|:----:|:---:|
Sham_oxy versus Sham_Sal in Condition |  61 |  54  | 105 |
SNI_oxy versus Sham_Sal in Condition  |   2 |   1  |   3 |
SNI_Sal versus Sham_Sal in Condition  | 159 |  87  | 246 |


The paper mentions differentially expressed genes in three brain regions : the NAc, mPFC and VTA. Briefly explain what these 3 regions are.

**NAc (Nucleus Accumbens)** \
A key structure of the brain's reward circuitry, located in the ventral striatum. It plays a central role in processing reward, motivation, and reinforcement, and is heavily implicated in addiction and the reinforcing effects of drugs of abuse, including opioids. It's also increasingly studied for its role in chronic pain, since pain and reward circuits interact (chronic pain can blunt reward processing, and the NAc is thought to contribute to pain-induced anhedonia).

**mPFC (medial Prefrontal Cortex)** \
Part of the prefrontal cortex, involved in executive function, decision-making, emotional regulation, and the cognitive/affective components of pain (how pain is perceived and experienced, beyond just the raw sensory signal). It has strong connections to both the NAc and other limbic structures, making it relevant to the emotional and motivational aspects of chronic pain and addiction-related behavior.

**VTA (Ventral Tegmental Area)** \
Located in the midbrain, this is the origin of the brain's main dopaminergic pathway (the mesolimbic dopamine system), projecting to the NAc and mPFC among other targets. It's the key source of dopamine signaling that drives reward and motivation, and is central to both the rewarding effects of opioids and to how chronic pain can dysregulate the reward system.

Together, the VTA, NAc, and mPFC form the core of the mesocorticolimbic dopamine pathway, the brain's reward circuit. Given the study is looking at an opioid (oxycodone) in the context of a neuropathic pain model (SNI), examining transcriptional changes across these three interconnected reward-processing regions makes sense, pain and opioid treatment are both expected to act on this circuit, affecting motivation, mood, and reward processing, not just sensory pain pathways.

Is there anyway from the paper and the material and methods for us to know which genes are included in these regions?

Once you have your list of differentially expressed genes, do you think just communicating those to the biologists would be sufficient? What does the publication state?

Please reproduce the Venn Diagram from Figure 3, not taking into account the brain regions but just the contrasts mentionned.